Searching for data augmentation strategies, we decided to stick to the synonym replacement, namely, the **random swap** strategy. 


– random insertion (This is an english example => This is an alternative english example)

– random swap (This is an english example => This is an english illustration)

– random deletion (This is an english example => This is an example)

The following code snippet replaces one word (either noun or verb) in every sentence. Here, I'm using the wordnet synsets to find the synonyms. The examples are only shown in English.

In [2]:
import nltk
nltk.download("wordnet")
from nltk.corpus import wordnet as wn
nltk.download('omw-1.4')

import pandas as pd
import random
import spacy

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\white\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\white\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
nlp = spacy.load("en_core_web_sm")

In [4]:
pos_tags = ["VERB", "NOUN"]

In [5]:
def get_synonyms_pt(word):
    synonyms = set()
    for syn in wn.synsets(word, lang="por"):  
        for lemma in syn.lemmas(lang="por"):
            if lemma.name().replace("_", " ") != word.lower():
                synonyms.add(lemma.name().replace("_", " "))  
    return list(synonyms)

def get_synonyms_en(word):
    synonyms = set()
    for syn in wn.synsets(word):  
        for lemma in syn.lemmas():
            if lemma.name().replace("_", " ") != word.lower():
                synonyms.add(lemma.name().replace("_", " ")) 
    return list(synonyms)

def synonym_replacement_pt(sentence):
    words = sentence.split()
    replaceable_words = [word for word in words if get_synonyms_pt(word)]  # Words with synonyms

    if not replaceable_words:
        return sentence  # If there are no synonyms available, return original sentence

    word_to_replace = random.choice(replaceable_words)
    synonym = random.choice(get_synonyms_pt(word_to_replace))

    new_sentence = [synonym if word == word_to_replace else word for word in words]
    return " ".join(new_sentence)

def synonym_replacement_en(sentence):
    words = sentence.split()
    doc = nlp(sentence)
    words_of_pos = [token for token in doc if token.pos_ in pos_tags]
    words_of_pos = [str(word) for word in words_of_pos]
    replaceable_words = [word for word in words_of_pos if get_synonyms_en(str(word))]  # Words with synonyms

    if not replaceable_words:
        return sentence  # If there are no synonyms available, return original sentence

    word_to_replace = random.choice(replaceable_words)

    synonym = random.choice(get_synonyms_en(word_to_replace))

    new_sentence = [synonym if word == word_to_replace else word for word in words]
    return " ".join(new_sentence)


In [6]:
synonym_replacement_en("I cannot love you")

'I cannot have it away you'

In [7]:
dev_df = pd.read_csv("C:/Users/white/Desktop/ANLP Project/SubTaskA/Data/dev.csv")

In [8]:
get_synonyms_en("now")  #Demonstrating a random synset for an english word

['instantly',
 'immediately',
 'today',
 'at present',
 'like a shot',
 'directly',
 'nowadays',
 'straightaway',
 'right away',
 'at once',
 'straight off',
 'forthwith']

In [9]:
get_synonyms_pt("feliz")  #Demonstrating a random synset for a portuguese word

['contente', 'jovial', 'alegre', 'grato', 'risonho']

In [10]:
dev_df_eng = dev_df.iloc[:466]

In [11]:
text_en= dev_df_eng["Target"].apply(synonym_replacement_en)

In [12]:
dev_df_eng

,ID,Language,MWE,Previous,Target,Next
0,3652,EN,high life,"Does the plumbing predictably rebel, creating ...",Are these interruptions of the good life a nec...,"Not at all , says James von Klemperer, preside..."
1,11103,EN,high life,Let’s be honest – we would be chuffed if our b...,But for Australian fashion designer Abby Kheir...,"Speaking to the Daily Mail , the owner of Abys..."
2,84346,EN,high life,I already have the winning ticket.,"With that, I will be enjoying the pleasures of...","Charles Selle is a former News-Sun reporter, p..."
3,56279,EN,high life,"There were signs everywhere of cosy, comfortab...","Fendi offered swaddling, belted coats resembli...",Shawl collars and enveloping dressing-gown sty...
4,17886,EN,high life,Yet one thing is clear.,"Rick Ross and Diddy exemplify the high life, t...",And who knows -- maybe we'll see Puff on his a...
...,...,...,...,...,...,...
461,95264,EN,life vest,It also didn't have a life preserver or other ...,The man who rented Rivera the boat has said sh...,The suit also said there weren't any signs in ...
462,39232,EN,life vest,While the color palette stays vibrant througho...,The migrants from Libya are going on a boat wi...,At the same time it's a better decision for th...
463,77031,EN,life vest,He plumbs its depths as he contemplates notion...,"He knows its history, its contours, its moods ...","The Susquehanna is companion, teacher and muse."
464,76111,EN,life vest,Those uninsured homeowners will likely have to...,Wright said such federal emergency help should...,Wright said that nationally there are about 10...


In [13]:
dev_df_eng = dev_df_eng.assign(replaced=text_en)

In [20]:
for replace, target in zip(dev_df_eng["replaced"][:10], dev_df_eng["Target"][:10]):
    print(f"Replace: {replace} ")
    print('\n')
    print(f"Target: {target}")
    print('\n')

Replace: Are these interruptions of the good life sentence a necessary condition of the high life? 


Target: Are these interruptions of the good life a necessary condition of the high life?


Replace: But for Australian fashion designer Abby Kheir, there's no reason not to treat her employees to a predilection of the high life all-year round. 


Target: But for Australian fashion designer Abby Kheir, there's no reason not to treat her employees to a taste of the high life all-year round.


Replace: With that, I will be enjoying the pleasures of the high life knowing I earned money the hard way: Gambling with a bit of mega luck. 


Target: With that, I will be enjoying the pleasures of the high life knowing I earned money the hard way: Gambling with a bit of mega luck.


Replace: Fendi offered swaddling, belted coats resembling dressing gowns over warm trousers, all in materials promising soft rest as much as the adventurous high life. 


Target: Fendi offered swaddling, belted coats r